In [1]:
import matplotlib.pyplot as plt 
import numpy as np 
import os 
import pandas as pd # 

In [2]:
df = pd.read_csv('match_data_50_tourns_modified.csv')

In [3]:
df.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,242,32,17,32,17,5,0,0.0,1.000000,5808
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,4,245,118,808,430,2,5,1.0,0.285714,5808
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,215,101,49,336,147,5,3,0.0,0.625000,5808
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,282,91,35,235,97,5,2,0.0,0.714286,5808
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,61,281,170,1112,649,2,5,1.0,0.285714,5808


In [4]:
n = round(len(df) / 6)
df_holdout = df.tail(n)
df=df.iloc[:-n]

### kNN regression with statistical features
We pick only the statistical features for linear regression, elo ratings and match and frame win predictions from the elo ratings are dropped.

In [5]:
features_to_remove = ['player1', 'player2', 'player1_elo','player2_elo','elo_match_win_rate','elo_frame_win_rate',
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [6]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, root_mean_squared_error
import numpy as np

In [7]:
tscv = TimeSeriesSplit(n_splits=5)

knn = KNeighborsRegressor()
param_grid = {'n_neighbors': range(1, 10)}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': range(1, 10)},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

We select the best model from the grid search cross validation. Since we are using a weighted RMSE, we need to define a custom loop to compute WRMSE and pick the model with lowest value.

In [8]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

In [9]:
cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index] 

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'n_neighbors': 9}
Best weighted average RMSE: 0.2919090033498773


In [12]:
# Sanity check, to verify that n_neihbors=9 indeed gives a WRMSE of 0.2919090033498773
tscv = TimeSeriesSplit(n_splits=5)

rmse_list = []
train_sizes = []

for train_index, test_index in tscv.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Test size:  {len(test_index)}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = KNeighborsRegressor(n_neighbors=9)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    rmse = root_mean_squared_error(y_test, y_pred)
    rmse_train = root_mean_squared_error(y_train, y_train_pred)
    rmse_list.append(rmse)
    train_sizes.append(len(train_index))

    print(f"Train RMSE: {rmse_train} ")
    print(f"Fold RMSE: {rmse} \n")

  Train size: 705
  Test size:  702
Train RMSE: 0.23882711579348065 
Fold RMSE: 0.33452178189509424 

  Train size: 1407
  Test size:  702
Train RMSE: 0.26536272365359564 
Fold RMSE: 0.2771093679997995 

  Train size: 2109
  Test size:  702
Train RMSE: 0.25752095633663236 
Fold RMSE: 0.28918321357081633 

  Train size: 2811
  Test size:  702
Train RMSE: 0.2573551658153741 
Fold RMSE: 0.2620156831305273 

  Train size: 3513
  Test size:  702
Train RMSE: 0.2519119754822649 
Fold RMSE: 0.3148409395422523 



In [13]:
weighted_rmse = np.average(rmse_list, weights=train_sizes)
print(f"Weighted Avg RMSE:   {weighted_rmse:.4f}")

Weighted Avg RMSE:   0.2919


### kNN with statistical features based on player statistic differences

In [14]:
dfm = df

In [15]:
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']
dfm.fillna(0.5, inplace=True)
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']



In [16]:
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']

In [17]:
X = dfm[selected_features]
y = dfm['win_percentage']

In [18]:
tscv = TimeSeriesSplit(n_splits=5)

knn = KNeighborsRegressor()
param_grid = {'n_neighbors': range(1, 10)}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': range(1, 10)},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [19]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)
cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'n_neighbors': 9}
Best weighted average RMSE: 0.2939397972333141


### kNN regression with player elo ratings and features based on player statistic differences

In [20]:
selected_features = ['player1_elo','player2_elo','matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']

In [21]:
X = dfm[selected_features]
y = dfm['win_percentage']

In [22]:
tscv = TimeSeriesSplit(n_splits=5)

knn = KNeighborsRegressor()
param_grid = {'n_neighbors': range(1, 10)}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': range(1, 10)},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [23]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'n_neighbors': 9}
Best weighted average RMSE: 0.2842554482802658


### kNN on all available features

In [24]:
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [25]:
tscv = TimeSeriesSplit(n_splits=5)

knn = KNeighborsRegressor()
param_grid = {'n_neighbors': range(1, 10)}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': range(1, 10)},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [26]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'n_neighbors': 9}
Best weighted average RMSE: 0.29043024166519255


Even the best performing kNN model is not performing good enough compared to elo predictions or linear regression.